# Named runtime repair: proof before production TDD

This notebook tests candidate routing and execution-origin contracts before production edits. Rust cells exercise a source-grounded policy model, not a compiled replacement for the host. Python cells run real ipykernel background work, then test a proposed context carrier with an explicit host receipt model. Production tests must reproduce the actual APIs and remain authoritative for RED/GREEN.

The repairs preserve factory kernels, notebook/profile ownership, current process registration, generation, source/version and Running admission. No copied factory port helper or filesystem publication bypass is used. After these proofs, production changes will be made in isolated external-repository worktrees.

In [3]:
mod routing_proof {
    pub fn current(spec: &str, slot: &str) -> bool {
        matches!(spec, "xcpp" | "xcpp17" | "xcpp20" | "xcpp23" | "evcxr")
            || (spec == "python3" && slot.starts_with("native-profile:"))
    }
    pub fn candidate(spec: &str, slot: &str) -> bool {
        matches!(spec, "xcpp" | "xcpp17" | "xcpp20" | "xcpp23" | "evcxr")
            || (matches!(spec, "python3" | "deno") && slot.starts_with("native-profile:"))
    }
}
let routing_cases = [
    ("deno", "native-profile:v1:valid-owned-profile", false, true),
    ("deno", "notebook:example.ipynb#deno", false, false),
    ("python3", "native-profile:v1:valid-owned-profile", true, true),
    ("python3", "notebook:example.ipynb#python3", false, false),
    ("evcxr", "notebook:example.ipynb#evcxr", true, true),
    ("xcpp20", "notebook:example.ipynb#xcpp20", true, true),
    ("unknown", "native-profile:v1:untrusted", false, false),
];
for (spec, slot, before, after) in routing_cases {
    assert_eq!(routing_proof::current(spec, slot), before);
    assert_eq!(routing_proof::candidate(spec, slot), after);
    println!("{} | {} | current={} candidate={}", spec, slot, before, after);
}
println!("ROUTING_MODEL_PASS: named Deno changes; factory Deno stays legacy. Ownership is validated later by authoritative dispatch.");

deno | native-profile:v1:valid-owned-profile | current=false candidate=true


deno | notebook:example.ipynb#deno | current=false candidate=false


python3 | native-profile:v1:valid-owned-profile | current=true candidate=true


python3 | notebook:example.ipynb#python3 | current=false candidate=false


evcxr | notebook:example.ipynb#evcxr | current=true candidate=true


xcpp20 | notebook:example.ipynb#xcpp20 | current=true candidate=true


unknown | native-profile:v1:untrusted | current=false candidate=false


ROUTING_MODEL_PASS: named Deno changes; factory Deno stays legacy. Ownership is validated later by authoritative dispatch.


In [2]:
import asyncio,threading,contextvars,json,os,sys
from dataclasses import dataclass,asdict
@dataclass(frozen=True)
class Origin:
 notebook: str
 profile: str
 process_generation: int
 source_version: int
 request_id: str
origin_var=contextvars.ContextVar('proof_execution_origin',default=None)
proof_kernel=get_ipython().kernel
origin_A=Origin('proof-notebook','proof-python',1,1,proof_kernel.get_parent()['header']['msg_id'])
origin_var.set(origin_A)
proof_async_gate=asyncio.Event(); proof_thread_gate=threading.Event()
proof_records={}
async def proof_async_worker():
 await proof_async_gate.wait()
 proof_records['async_origin']=origin_var.get()
 proof_records['async_parent']=proof_kernel.get_parent()['header']['msg_id']
def proof_thread_worker():
 proof_thread_gate.wait()
 proof_records['thread_origin']=origin_var.get()
 proof_records['thread_parent']=proof_kernel.get_parent()['header']['msg_id']
proof_task=asyncio.create_task(proof_async_worker())
proof_thread=threading.Thread(target=proof_thread_worker,daemon=True);proof_thread.start()
print(json.dumps({'phase':'A','origin':asdict(origin_A),'pid':os.getpid(),'executable':sys.executable,'scope':'carrier experiment; no port publication attempted'}))

{"phase": "A", "origin": {"notebook": "proof-notebook", "profile": "proof-python", "process_generation": 1, "source_version": 1, "request_id": "f12219f2-ff9b-41b3-9965-413c30b80d5e"}, "pid": 6865, "executable": "/Users/kevintruong/.spur/jupyter/venv-duckdb-2.0/bin/python", "scope": "carrier experiment; no port publication attempted"}


In [3]:
origin_B=Origin('proof-notebook','proof-python',1,2,proof_kernel.get_parent()['header']['msg_id'])
origin_var.set(origin_B)
proof_async_gate.set();proof_thread_gate.set()
await proof_task
proof_thread.join()
def model_admit(carrier,current_running):
 return carrier is not None and carrier==current_running
carrier_observation={
 'phase_A':asdict(origin_A),'phase_B':asdict(origin_B),
 'async_origin_preserved':proof_records['async_origin']==origin_A,
 'unbound_thread_has_no_origin':proof_records['thread_origin'] is None,
 'thread_latest_parent_is_B':proof_records['thread_parent']==origin_B.request_id,
 'current_B_admitted':model_admit(origin_var.get(),origin_B),
 'late_async_A_rejected_while_B_running':not model_admit(proof_records['async_origin'],origin_B),
 'unbound_thread_rejected':not model_admit(proof_records['thread_origin'],origin_B)}
assert all(carrier_observation[k] for k in ['async_origin_preserved','unbound_thread_has_no_origin','thread_latest_parent_is_B','current_B_admitted','late_async_A_rejected_while_B_running','unbound_thread_rejected'])
print(json.dumps(carrier_observation,indent=2))

{
  "phase_A": {
    "notebook": "proof-notebook",
    "profile": "proof-python",
    "process_generation": 1,
    "source_version": 1,
    "request_id": "f12219f2-ff9b-41b3-9965-413c30b80d5e"
  },
  "phase_B": {
    "notebook": "proof-notebook",
    "profile": "proof-python",
    "process_generation": 1,
    "source_version": 2,
    "request_id": "fecd0948-eb85-42b8-bd2f-f16c04c7efbb"
  },
  "async_origin_preserved": true,
  "unbound_thread_has_no_origin": true,
  "thread_latest_parent_is_B": true,
  "current_B_admitted": true,
  "late_async_A_rejected_while_B_running": true,
  "unbound_thread_rejected": true
}


In [4]:
mod publication_proof {
 #[derive(Clone,Debug,PartialEq,Eq)]
 pub struct Stamp {pub notebook:&'static str,pub profile:&'static str,pub generation:u64,pub source_version:u64,pub run:&'static str,pub fingerprint:&'static str,pub registration:&'static str}
 pub fn admit(language:&str,qualified:bool,carrier:Option<&Stamp>,current:&Stamp,running:bool,cancelled:bool)->bool {
   matches!(language,"deno"|"python") && qualified && running && !cancelled && carrier==Some(current)
 }
}
use publication_proof::{Stamp,admit};
let current=Stamp{notebook:"notebook-A",profile:"profile-A",generation:1,source_version:2,run:"run-B",fingerprint:"inputs-B",registration:"registration-B"};
assert!(admit("python",true,Some(&current),&current,true,false));
assert!(admit("deno",true,Some(&current),&current,true,false));
assert!(!admit("python",false,Some(&current),&current,true,false));
assert!(!admit("python",true,None,&current,true,false));
assert!(!admit("python",true,Some(&current),&current,false,false));
assert!(!admit("python",true,Some(&current),&current,true,true));
let mut invalid=Vec::new();
let mut x=current.clone();x.notebook="notebook-other";invalid.push(x);
let mut x=current.clone();x.profile="profile-other";invalid.push(x);
let mut x=current.clone();x.generation=2;invalid.push(x);
let mut x=current.clone();x.source_version=1;invalid.push(x);
let mut x=current.clone();x.run="run-A";invalid.push(x);
let mut x=current.clone();x.fingerprint="inputs-A";invalid.push(x);
let mut x=current.clone();x.registration="registration-A";invalid.push(x);
for stamp in &invalid { assert!(!admit("python",true,Some(stamp),&current,true,false)); }
println!("PUBLICATION_MODEL_PASS: both qualified languages admitted; {} identity mutations, absent carrier, stale Running, cancellation and missing qualification rejected. No production host IPC is exercised.",invalid.len());

PUBLICATION_MODEL_PASS: both qualified languages admitted; 7 identity mutations, absent carrier, stale Running, cancellation and missing qualification rejected. No production host IPC is exercised.


In [4]:
from pathlib import Path
import hashlib,subprocess
source_root=Path('/Users/kevintruong/project/spur-notebook')
source_files=['src/dag/engine.rs','src/mcp/tools/native_profile_lifecycle.rs','jute-notebook/src-tauri/src/kernel_config/target.rs','jute-notebook/src-tauri/src/kernel_config/runtime.rs','jute-notebook/src-tauri/src/kernel_config/runtime_provenance.rs','src/dag/deno_publication_bridge.rs']
proof_sources={'head':subprocess.run(['git','-C',str(source_root),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip(),'files':{p:hashlib.sha256((source_root/p).read_bytes()).hexdigest() for p in source_files}}
print(json.dumps(proof_sources,indent=2))

{
  "head": "b5885ef86b939680b7803c236ee81005beac8aa9",
  "files": {
    "src/dag/engine.rs": "3cf79ef72af674c326ec95897c21a53987046423b4246cf305e00830fa379d43",
    "src/mcp/tools/native_profile_lifecycle.rs": "a713df60c13498bad45b1f00cc9ca23eac44bad3100456ea82576837cd1af902",
    "jute-notebook/src-tauri/src/kernel_config/target.rs": "bd5a62dba8d4ae08b5b86e90019a6d11868c06c2f6c04206687b10f2aab2674e",
    "jute-notebook/src-tauri/src/kernel_config/runtime.rs": "35f210beefd47e901cb2d92d5b6b6aadea8686ec814b9e96bcf2343233e3148e",
    "jute-notebook/src-tauri/src/kernel_config/runtime_provenance.rs": "46b7176dab1191b9dd42caf17f92a6125ed7096a2481a2f8140d8732c9617643",
    "src/dag/deno_publication_bridge.rs": "75ad16df11c8072589f218fa760c6a94e6f02a3313e9fea031433e33392f1e11"
  }
}


In [5]:
# Collector contract experiment before C1 implementation. Host API receipt is explicit.
import hashlib,subprocess,json
from pathlib import Path
request=json.loads("{\"repository\":\"duckdb/duckdb\",\"number\":28731,\"base_sha\":\"629e395a854d72dcb1f414e1a0451c68ac0e9f1f\",\"head_sha\":\"d29b26bb92ac940a0807be0f43cc385b31623839\",\"updated_at\":\"2026-11-18T09:30:21Z\"}")
repo=Path('/Users/kevintruong/project/duckdb')
def git_bytes(*args):
 return subprocess.run(['git','-C',str(repo),*args],capture_output=True,check=True).stdout
for revision in [request['base_sha'],request['head_sha']]:
 assert git_bytes('cat-file','-t',revision).strip()==b'commit'
raw_tree=git_bytes('ls-tree','-r','-z','--full-tree',request['head_sha'])
raw_diff=git_bytes('diff','--name-status','-z',request['base_sha'],request['head_sha'])
fixture_path=Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-spur-review-ux/design-data.json')
fixture=json.loads(fixture_path.read_text())
def context_usable(pr,context):
 return pr['head_sha']==context['head'] and pr['base_sha']==context['base']
assert context_usable(request,fixture)
changed={**request,'head_sha':request['base_sha']}
assert not context_usable(changed,fixture)
provider_bytes={'tree':hashlib.sha256(raw_tree).hexdigest(),'diff':hashlib.sha256(raw_diff).hexdigest(),'context':hashlib.sha256(fixture_path.read_bytes()).hexdigest()}
run_core={'schema':'spur.review.run/v1','request':request,'providers':provider_bytes}
run_id=hashlib.sha256(json.dumps(run_core,sort_keys=True,separators=(',',':')).encode()).hexdigest()
assert run_id==hashlib.sha256(json.dumps(run_core,sort_keys=True,separators=(',',':')).encode()).hexdigest()
changed_core={**run_core,'request':changed}
assert run_id!=hashlib.sha256(json.dumps(changed_core,sort_keys=True,separators=(',',':')).encode()).hexdigest()
collector_proof={'run_id':run_id,'tracked_entries':raw_tree.count(b'\x00'),'changed_files':len(fixture['files']),'provider_hashes':provider_bytes,'context_refs_valid':all(row['evidence'] in fixture['evidence'] for context in fixture['contexts'].values() for row in context['rows']),'same_input_same_identity':True,'changed_head_changes_identity':True,'mixed_context_rejected':True,'context_discovery':'captured typed neighborhood, not a full semantic repo link','live_named_pipeline':False}
assert collector_proof['tracked_entries']==18342 and collector_proof['context_refs_valid']
print(json.dumps(collector_proof,indent=2))

{
  "run_id": "c62258ed1d00ec67e34638bd0891934a46216f37911e3ad651569da7a202e6c3",
  "tracked_entries": 18342,
  "changed_files": 5,
  "provider_hashes": {
    "tree": "b10a5991d6a6ed5d05f120cde75349562623a8c3ea864be8a932584bcfb033f9",
    "diff": "d2cff228c75168ee4c6bd8f0389c18b98a7708418d872a2f187b37893cfe0899",
    "context": "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2"
  },
  "context_refs_valid": true,
  "same_input_same_identity": true,
  "changed_head_changes_identity": true,
  "mixed_context_rejected": true,
  "context_discovery": "captured typed neighborhood, not a full semantic repo link",
  "live_named_pipeline": false
}


In [6]:
import subprocess,json,hashlib,sys
from pathlib import Path
client_worktree=Path('/Users/kevintruong/project/SpurCode/.spur/worktrees/review-python-bd-2sjn')
client_test_inputs={str(p.relative_to(client_worktree)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((client_worktree/'tests/python_project_ports').rglob('*.py'))}
client_command=[sys.executable,'-m','unittest','discover','-s','tests/python_project_ports','-v']
client_result=subprocess.run(client_command,cwd=client_worktree,capture_output=True,text=True)
client_after={str(p.relative_to(client_worktree)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((client_worktree/'tests/python_project_ports').rglob('*.py'))}
client_test_receipt={'command':client_command,'cwd':str(client_worktree),'exit':client_result.returncode,'inputs':client_test_inputs,'test_sources_stable':client_after==client_test_inputs,'stdout':client_result.stdout,'stderr':client_result.stderr,'qualification':'Python client/evaluator only; Rust canonical host wiring is a separate test'}
assert client_test_receipt['test_sources_stable']
print(json.dumps(client_test_receipt,indent=2))
assert client_result.returncode==0

{
  "command": [
    "/Users/kevintruong/.spur/jupyter/venv-duckdb-2.0/bin/python",
    "-m",
    "unittest",
    "discover",
    "-s",
    "tests/python_project_ports",
    "-v"
  ],
  "cwd": "/Users/kevintruong/project/SpurCode/.spur/worktrees/review-python-bd-2sjn",
  "exit": 0,
  "inputs": {
    "tests/python_project_ports/test_ipykernel.py": "aecab22db4623a8062f202a73a2864cb1da3cc568f0d8ac038e421423cd88139",
    "tests/python_project_ports/test_project_ports.py": "048e3c2d33a38996eeb51ff3009f7ac26235122658eea3c61af3ac14e6fd16f5"
  },
  "test_sources_stable": true,
  "stdout": "",
  "stderr": "test_async_origin_stays_A_and_plain_thread_has_no_origin_during_B (test_ipykernel.RealEvaluator.test_async_origin_stays_A_and_plain_thread_has_no_origin_during_B) ... [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-pro

In [8]:
# C0 RED: execute real-Git fixtures before collector implementation.
import subprocess,json,pathlib,time
collector_root=pathlib.Path('/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku')
collector_cmd=[__import__('sys').executable,'-m','unittest','discover','-s','app_gallery/spur-review/tests','-v']
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
collector_receipt={'phase':'RED','command':collector_cmd,'cwd':str(collector_root),'exit':collector_proc.returncode,'stdout':collector_proc.stdout,'stderr':collector_proc.stderr}
collector_art=collector_root/'docs/research/artifacts/2026-11-18-review-collector-bd-117o'
collector_art.mkdir(parents=True,exist_ok=True)
(collector_art/'red.json').write_text(json.dumps(collector_receipt,indent=2))
assert collector_proc.returncode==1 and 'immutable review collector is not implemented' in collector_proc.stderr
assert 'Ran 12 tests' in collector_proc.stderr and 'FAILED (failures=14)' in collector_proc.stderr
print(json.dumps({'phase':'RED','exit':collector_proc.returncode,'tests':12,'failures_including_subtests':14,'reason':'immutable collector missing','receipt':str(collector_art/'red.json')},indent=2))

{
  "phase": "RED",
  "exit": 1,
  "tests": 12,
  "failures_including_subtests": 14,
  "reason": "immutable collector missing",
  "receipt": "/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku/docs/research/artifacts/2026-11-18-review-collector-bd-117o/red.json"
}


In [9]:
# C0 GREEN: same real-Git fixture command after the minimal implementation.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
collector_receipt={'phase':'GREEN','command':collector_cmd,'cwd':str(collector_root),'exit':collector_proc.returncode,'stdout':collector_proc.stdout,'stderr':collector_proc.stderr}
(collector_art/'green.json').write_text(json.dumps(collector_receipt,indent=2))
print(collector_proc.stderr)
assert collector_proc.returncode==0
print(json.dumps({'phase':'GREEN','exit':0,'receipt':str(collector_art/'green.json')},indent=2))

test_broken_context_evidence_reference_is_rejected (test_collector.CollectorTests.test_broken_context_evidence_reference_is_rejected) ... ok
test_changed_head_cannot_reuse_old_context (test_collector.CollectorTests.test_changed_head_cannot_reuse_old_context) ... ok
test_different_valid_head_gets_different_run (test_collector.CollectorTests.test_different_valid_head_gets_different_run) ... ok
test_foreign_witness_revision_is_rejected (test_collector.CollectorTests.test_foreign_witness_revision_is_rejected) ... ok
test_full_tracked_inventory_contains_unchanged_context (test_collector.CollectorTests.test_full_tracked_inventory_contains_unchanged_context) ... ok
test_partial_provider_is_explicit_and_not_final_ready (test_collector.CollectorTests.test_partial_provider_is_explicit_and_not_final_ready) ... ok
test_provider_byte_change_changes_run_identity (test_collector.CollectorTests.test_provider_byte_change_changes_run_identity) ... ok
test_rename_maps_both_paths_without_losing_base_blob 

In [23]:
# Post-TDD grounding: production collector against the pinned DuckDB PR.
import importlib.util
collector_spec=importlib.util.spec_from_file_location('review_collector',collector_root/'app_gallery/spur-review/collector.py')
collector_module=importlib.util.module_from_spec(collector_spec)
collector_spec.loader.exec_module(collector_module)
real_request={'repository':'duckdb/duckdb','number':28731,'base_sha':'629e395a854d72dcb1f414e1a0451c68ac0e9f1f','head_sha':'d29b26bb92ac940a0807be0f43cc385b31623839'}
real_context=pathlib.Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-spur-review-ux/design-data.json').read_bytes()
# Captured native slices lack blob IDs. Verify every retained line against the
# pinned Git blob before deriving a new provider artifact; preserve original bytes.
qualified_context=json.loads(real_context)
qualified_count=0
for ev in qualified_context['evidence'].values():
 if 'witness_blob_oid' not in ev:
  oid=subprocess.run(['git','-C','/Users/kevintruong/project/duckdb','rev-parse',ev['revision']+':'+ev['witness_path']],capture_output=True,text=True,check=True).stdout.strip()
  lines=subprocess.run(['git','-C','/Users/kevintruong/project/duckdb','show',oid],capture_output=True,text=True,check=True).stdout.splitlines()
  assert ev.get('lines') and all(lines[line['n']-1]==line['text'] for line in ev['lines'])
  ev['witness_blob_oid']=oid
  qualified_count+=1
# One legacy row used a relationship description as witness_text.
# Retain that annotation separately and bind the source quote to its exact anchor.
annotation=qualified_context['evidence']['context:319']
annotation['relationship_description']=annotation['witness_text']
annotation_source=subprocess.run(['git','-C','/Users/kevintruong/project/duckdb','show',annotation['witness_blob_oid']],capture_output=True,text=True,check=True).stdout.splitlines()
annotation['witness_text']=annotation_source[annotation['witness_line']-1]
qualified_context['coverage']={'link_scope':'captured_two_file_syntax_and_candidate_context','repository_inventory':'full_tracked_head','semantic_repository_linking':False}
qualified_bytes=json.dumps(qualified_context,sort_keys=True).encode()
real_packet=collector_module.collect_review('/Users/kevintruong/project/duckdb',real_request,{'graph':{'status':'ready','artifact':qualified_bytes},'history':{'status':'unavailable','reason':'captured graph context only; current history provider not supplied'}})
assert len(real_packet['tracked_files'])==18342 and len(real_packet['changes'])==5
assert real_packet['state']=='partial'
assert real_packet['providers']['graph']['data']==qualified_context
real_receipt={'run_id':real_packet['run_id'],'tracked_entries':len(real_packet['tracked_files']),'changes':len(real_packet['changes']),'evidence_refs':len(real_packet['providers']['graph']['data']['evidence']),'state':real_packet['state'],'graph_semantics':'source references remain candidates, not compiler-proven calls','named_pipeline':False,'qualified_missing_blob_rows':qualified_count,'original_artifact_sha256':hashlib.sha256(real_context).hexdigest(),'qualified_artifact_sha256':hashlib.sha256(qualified_bytes).hexdigest()}
(collector_art/'real-repository.json').write_text(json.dumps(real_receipt,indent=2))
print(json.dumps(real_receipt,indent=2))

{
  "run_id": "f74b9037eaa0717ebc6ae285bebea5a2c63fa2ba0a46d0ad6d9a7b256f3fd574",
  "tracked_entries": 18342,
  "changes": 5,
  "evidence_refs": 21,
  "state": "partial",
  "graph_semantics": "source references remain candidates, not compiler-proven calls",
  "named_pipeline": false,
  "qualified_missing_blob_rows": 4,
  "original_artifact_sha256": "b8e1a021c9460ffdbe0882c8a3363cdf99df9569d2f91590c2554707d965c0d2",
  "qualified_artifact_sha256": "0300685487e43a6acb78b4f2043e552739e3817c1e19e744399efe623b1377f0"
}


In [12]:
# C0 hardening RED: a provider must not imply complete coverage by omission.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'coverage-red.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
assert collector_proc.returncode==1 and 'ValueError not raised' in collector_proc.stderr and 'FAILED (failures=1)' in collector_proc.stderr
print(collector_proc.stderr[-450:])

/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku/app_gallery/spur-review/tests/test_collector.py", line 112, in test_graph_provider_requires_explicit_coverage
    with self.assertRaisesRegex(ValueError, "coverage"):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: ValueError not raised

----------------------------------------------------------------------
Ran 13 tests in 5.697s

FAILED (failures=1)



In [13]:
# C0 hardening GREEN uses the identical test command.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'coverage-green.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
print(collector_proc.stderr)
assert collector_proc.returncode==0

test_broken_context_evidence_reference_is_rejected (test_collector.CollectorTests.test_broken_context_evidence_reference_is_rejected) ... ok
test_changed_head_cannot_reuse_old_context (test_collector.CollectorTests.test_changed_head_cannot_reuse_old_context) ... ok
test_different_valid_head_gets_different_run (test_collector.CollectorTests.test_different_valid_head_gets_different_run) ... ok
test_foreign_witness_revision_is_rejected (test_collector.CollectorTests.test_foreign_witness_revision_is_rejected) ... ok
test_full_tracked_inventory_contains_unchanged_context (test_collector.CollectorTests.test_full_tracked_inventory_contains_unchanged_context) ... ok
test_graph_provider_requires_explicit_coverage (test_collector.CollectorTests.test_graph_provider_requires_explicit_coverage) ... ok
test_partial_provider_is_explicit_and_not_final_ready (test_collector.CollectorTests.test_partial_provider_is_explicit_and_not_final_ready) ... ok
test_provider_byte_change_changes_run_identity (test_

In [24]:
# Provider experiment before named-app integration: existing binaries, no install.
import duckdb,pyarrow as pa
assert duckdb.__version__=='2.0.0'
review_engine=duckdb.connect(config={'allow_unsigned_extensions':True,'autoload_known_extensions':False,'autoinstall_known_extensions':False})
extension_receipts=json.loads(pathlib.Path('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-17-review-extensions-poc/extension-binaries.json').read_text())
for binary in extension_receipts:
 binary_path=pathlib.Path(binary['path'])
 assert hashlib.sha256(binary_path.read_bytes()).hexdigest()==binary['sha256']
 review_engine.load_extension(str(binary_path))
review_engine.execute("SET active_grammar_extensions = ['duckpgq']")
history_data={'base':real_request['base_sha'],'head':real_request['head_sha'],'coverage':{'scope':'changed_cpp_file_history_at_base','current_assignments':'unknown'},'history':[]}
for change in real_packet['changes']:
 if change['path'].endswith('.cpp') and change['base_path']:
  uri=review_engine.execute('SELECT git_uri(?,?,?)',['/Users/kevintruong/project/duckdb',change['base_path'],real_request['base_sha']]).fetchone()[0]
  rows=review_engine.execute('SELECT author_name,count(*) AS commits,max(author_date)::VARCHAR AS latest FROM git_log(?) GROUP BY author_name ORDER BY commits DESC,author_name',[uri]).fetchall()
  history_data['history'].append({'path':change['base_path'],'revision':real_request['base_sha'],'authors':[{'name':r[0],'commits':r[1],'latest':r[2],'meaning':'historical contributor, not current assignment'} for r in rows]})
assert len(history_data['history'])==2 and all(item['authors'] for item in history_data['history'])
prepared_packet=collector_module.collect_review('/Users/kevintruong/project/duckdb',real_request,{'graph':{'status':'ready','artifact':qualified_bytes},'history':{'status':'ready','artifact':json.dumps(history_data,sort_keys=True).encode()}})
assert prepared_packet['state']=='ready'
packet_bytes=json.dumps(prepared_packet,sort_keys=True,separators=(',',':'))
packet_table=pa.table({'run_id':[prepared_packet['run_id']],'packet_json':[packet_bytes]})
buffer=pa.BufferOutputStream()
with pa.ipc.new_stream(buffer,packet_table.schema) as writer:
 writer.write_table(packet_table)
assert pa.ipc.open_stream(buffer.getvalue()).read_all().equals(packet_table)
(collector_art/'qualified-context.json').write_bytes(qualified_bytes)
(collector_art/'history-provider.json').write_text(json.dumps(history_data,indent=2))
provider_proof={'run_id':prepared_packet['run_id'],'tracked_files':len(prepared_packet['tracked_files']),'changed_files':len(prepared_packet['changes']),'history_files':len(history_data['history']),'state':'ready packet with explicit limited semantic scope','grammar':review_engine.sql("SELECT current_setting('active_grammar_extensions')").fetchall()[0][0],'arrow_roundtrip':True,'arrow_bytes':len(buffer.getvalue()),'live_named_publication':False,'current_stakeholders':'unknown; historical contributors are available'}
(collector_art/'provider-proof.json').write_text(json.dumps(provider_proof,indent=2))
print(json.dumps(provider_proof,indent=2))

{
  "run_id": "c7d9b335b003d6530b505fc9bb24ab09daa5543a17ba9944570c169785fbde68",
  "tracked_files": 18342,
  "changed_files": 5,
  "history_files": 2,
  "state": "ready packet with explicit limited semantic scope",
  "grammar": [
    "duckpgq"
  ],
  "arrow_roundtrip": true,
  "arrow_bytes": 2700688,
  "live_named_publication": false,
  "current_stakeholders": "unknown; historical contributors are available"
}


In [17]:
# Citation hardening RED: a correct blob ID is insufficient for a forged snippet.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'citation-red.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
assert collector_proc.returncode==1 and 'FAILED (failures=3)' in collector_proc.stderr
print(collector_proc.stderr[-900:])

eError, "witness"):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: ValueError not raised

FAIL: test_retained_source_lines_are_checked_against_blob (test_collector.CollectorTests.test_retained_source_lines_are_checked_against_blob)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku/app_gallery/spur-review/tests/test_collector.py", line 136, in test_retained_source_lines_are_checked_against_blob
    with self.assertRaisesRegex(ValueError, "witness"):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: ValueError not raised

----------------------------------------------------------------------
Ran 17 tests in 5.998s

FAILED (failures=3)



In [18]:
# Independent-review RED: spans, graph references, identities and Git object kinds.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'review-red.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
assert collector_proc.returncode==1 and 'ValueError not raised' in collector_proc.stderr
print(collector_proc.stderr[-1300:])

ge_citations_require_evidence) (node='e1', edge='missing')
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku/app_gallery/spur-review/tests/test_collector.py", line 159, in test_node_and_edge_citations_require_evidence
    with self.assertRaisesRegex(ValueError, "evidence"):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: ValueError not raised

FAIL: test_retained_source_lines_are_checked_against_blob (test_collector.CollectorTests.test_retained_source_lines_are_checked_against_blob)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku/app_gallery/spur-review/tests/test_collector.py", line 136, in test_retained_source_lines_are_checked_against_blob
    with self.assertRaisesRegex(Value

In [19]:
# Citation/graph review GREEN: identical real Git command after integrity validation.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'review-green.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
print(collector_proc.stderr)
assert collector_proc.returncode==0

test_broken_context_evidence_reference_is_rejected (test_collector.CollectorTests.test_broken_context_evidence_reference_is_rejected) ... ok
test_changed_head_cannot_reuse_old_context (test_collector.CollectorTests.test_changed_head_cannot_reuse_old_context) ... ok
test_different_valid_head_gets_different_run (test_collector.CollectorTests.test_different_valid_head_gets_different_run) ... ok
test_foreign_witness_revision_is_rejected (test_collector.CollectorTests.test_foreign_witness_revision_is_rejected) ... ok
test_forged_witness_text_with_correct_blob_is_rejected (test_collector.CollectorTests.test_forged_witness_text_with_correct_blob_is_rejected) ... ok
test_full_tracked_inventory_contains_unchanged_context (test_collector.CollectorTests.test_full_tracked_inventory_contains_unchanged_context) ... ok
test_gitlink_stays_in_inventory_but_cannot_be_a_source_blob (test_collector.CollectorTests.test_gitlink_stays_in_inventory_but_cannot_be_a_source_blob) ... ok
test_graph_endpoints_and_

In [21]:
# Real evidence uses an anchor quote plus a wider retained source window.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'window-red.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
assert collector_proc.returncode==1 and 'valid anchor quote in exact context window was rejected' in collector_proc.stderr
print(collector_proc.stderr[-700:])

urce")
ValueError: witness text does not match the pinned source

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku/app_gallery/spur-review/tests/test_collector.py", line 165, in test_anchor_quote_can_be_smaller_than_retained_context_window
    self.fail(f"valid anchor quote in exact context window was rejected: {error}")
AssertionError: valid anchor quote in exact context window was rejected: witness text does not match the pinned source

----------------------------------------------------------------------
Ran 21 tests in 9.814s

FAILED (failures=1)



In [22]:
# Final citation integrity GREEN, including valid quotes in retained source windows.
collector_proc=subprocess.run(collector_cmd,cwd=collector_root,capture_output=True,text=True)
(collector_art/'citation-green.json').write_text(json.dumps({'command':collector_cmd,'exit':collector_proc.returncode,'stderr':collector_proc.stderr},indent=2))
print(collector_proc.stderr)
assert collector_proc.returncode==0

test_anchor_quote_can_be_smaller_than_retained_context_window (test_collector.CollectorTests.test_anchor_quote_can_be_smaller_than_retained_context_window) ... ok
test_broken_context_evidence_reference_is_rejected (test_collector.CollectorTests.test_broken_context_evidence_reference_is_rejected) ... ok
test_changed_head_cannot_reuse_old_context (test_collector.CollectorTests.test_changed_head_cannot_reuse_old_context) ... ok
test_different_valid_head_gets_different_run (test_collector.CollectorTests.test_different_valid_head_gets_different_run) ... ok
test_foreign_witness_revision_is_rejected (test_collector.CollectorTests.test_foreign_witness_revision_is_rejected) ... ok
test_forged_witness_text_with_correct_blob_is_rejected (test_collector.CollectorTests.test_forged_witness_text_with_correct_blob_is_rejected) ... ok
test_full_tracked_inventory_contains_unchanged_context (test_collector.CollectorTests.test_full_tracked_inventory_contains_unchanged_context) ... ok
test_gitlink_stays_in

In [25]:
# Independent D2 source verification using actual bundled Deno/Jupyter.
import subprocess,json,pathlib,hashlib
deno_read_root=pathlib.Path('/Users/kevintruong/project/SpurCode/.spur/worktrees/review-deno-read-bd-23p5')
deno_read_cmd=['/Users/kevintruong/.spur/jupyter/venv/bin/python','tests/deno_project/project_ports_probe.py','--endpoint','http://127.0.0.1:9/','--origin','{}','--deno','/Users/kevintruong/Applications/SpurLab.app/Contents/MacOS/deno','--recovery-only']
deno_read_source=deno_read_root/'jute-notebook/src-tauri/src/assets/deno/ports_project_v2.js'
deno_read_hash=hashlib.sha256(deno_read_source.read_bytes()).hexdigest()
deno_read_result=subprocess.run(deno_read_cmd,cwd=deno_read_root,capture_output=True,text=True)
deno_read_receipt={'command':deno_read_cmd,'cwd':str(deno_read_root),'exit':deno_read_result.returncode,'stdout':deno_read_result.stdout,'stderr':deno_read_result.stderr,'source_sha256':deno_read_hash,'installed_app_updated':False,'production_bridge_qualified':False}
(collector_art/'deno-read-independent.json').write_text(json.dumps(deno_read_receipt,indent=2))
print(deno_read_result.stdout)
assert deno_read_result.returncode==0
assert hashlib.sha256(deno_read_source.read_bytes()).hexdigest()==deno_read_hash
print(json.dumps({'exit':0,'source_sha256':deno_read_hash,'scope':'actual helper/runtime, controlled HTTP peer; C1 production path pending'},indent=2))

{"status": "passed", "probe": "helper-http-reply-loss", "allocations": 1, "commits": 1, "read_failure": "read_unavailable", "arrow_reads": "passed", "production_qualification": false}

{
  "exit": 0,
  "source_sha256": "865a57749d5385008b33452d051a3b03e0c6ee71aefd7f23f2857b79c75ec9f9",
  "scope": "actual helper/runtime, controlled HTTP peer; C1 production path pending"
}


In [26]:
# Independent verification of C0 citation rework on its isolated branch.
citation_root=pathlib.Path('/Users/kevintruong/project/SpurCode/.spur/worktrees/review-collector-rework-bd-117o')
citation_cmd=[__import__('sys').executable,'-m','unittest','discover','-s','app_gallery/spur-review/tests','-v']
citation_run=subprocess.run(citation_cmd,cwd=citation_root,capture_output=True,text=True)
citation_receipt={'command':citation_cmd,'cwd':str(citation_root),'exit':citation_run.returncode,'stdout':citation_run.stdout,'stderr':citation_run.stderr,'source_sha256':hashlib.sha256((citation_root/'app_gallery/spur-review/collector.py').read_bytes()).hexdigest()}
(collector_art/'citation-rework-independent.json').write_text(json.dumps(citation_receipt,indent=2))
assert citation_run.returncode==0
citation_spec=importlib.util.spec_from_file_location('citation_collector',citation_root/'app_gallery/spur-review/collector.py')
citation_module=importlib.util.module_from_spec(citation_spec)
citation_spec.loader.exec_module(citation_module)
citation_packet=citation_module.collect_review('/Users/kevintruong/project/duckdb',real_request,{'graph':{'status':'ready','artifact':qualified_bytes},'history':{'status':'ready','artifact':json.dumps(history_data,sort_keys=True).encode()}})
assert citation_packet['run_id']==prepared_packet['run_id']
print(citation_run.stderr[-220:])
print(json.dumps({'physical_coordinate_rework':True,'real_repository_identity_unchanged':True,'run_id':citation_packet['run_id'],'source_sha256':citation_receipt['source_sha256']},indent=2))

change_pinned_collection (test_collector.CollectorTests.test_working_tree_edits_do_not_change_pinned_collection) ... ok

----------------------------------------------------------------------
Ran 27 tests in 13.126s

OK

{
  "physical_coordinate_rework": true,
  "real_repository_identity_unchanged": true,
  "run_id": "c7d9b335b003d6530b505fc9bb24ab09daa5543a17ba9944570c169785fbde68",
  "source_sha256": "f791ca632802fab838aef4769214b4eac88173c994cb9f2f2cf24f24f28aa5a7"
}


In [27]:
# Fresh checks on the integrated source after the documented EOF cleanup.
integration_root=pathlib.Path('/Users/kevintruong/project/SpurCode/.spur/worktrees/review-integration-bd-3oku')
integration_art=integration_root/'docs/research/artifacts/2026-11-18-review-candidate-bd-3oku'
integration_checks=[]
for suite in ['tests/python_project_ports','app_gallery/spur-review/tests']:
 command=[__import__('sys').executable,'-m','unittest','discover','-s',suite,'-v']
 result=subprocess.run(command,cwd=integration_root,capture_output=True,text=True)
 receipt={'suite':suite,'command':command,'cwd':str(integration_root),'exit':result.returncode,'stdout':result.stdout,'stderr':result.stderr}
 (integration_art/(suite.replace('/','-')+'.json')).write_text(json.dumps(receipt,indent=2))
 integration_checks.append({'suite':suite,'exit':result.returncode,'tail':result.stderr[-250:]})
 assert result.returncode==0,result.stderr
print(json.dumps(integration_checks,indent=2))

[
  {
    "suite": "tests/python_project_ports",
    "exit": 0,
    "tail": "_broken_carrier_cannot_qualify_from_version_strings (test_qualification.Qualification.test_broken_carrier_cannot_qualify_from_version_strings) ... ok\n\n----------------------------------------------------------------------\nRan 14 tests in 16.193s\n\nOK\n"
  },
  {
    "suite": "app_gallery/spur-review/tests",
    "exit": 0,
    "tail": "est_working_tree_edits_do_not_change_pinned_collection (test_collector.CollectorTests.test_working_tree_edits_do_not_change_pinned_collection) ... ok\n\n----------------------------------------------------------------------\nRan 27 tests in 32.722s\n\nOK\n"
  }
]


In [28]:
# Actual PR -> collector -> Arrow IPC -> bundled Deno -> review renderer.
# This verifies the consumer seam; the patched native host remains a separate gate.
import uuid, importlib.util, pyarrow as pa
candidate_root=pathlib.Path('/private/tmp/spur-review-bd-3oku-wuusb86g')
consumer_root=pathlib.Path('/Users/kevintruong/project/SpurCode/.spur/worktrees/review-consumer-bd-npyh')
collector_spec=importlib.util.spec_from_file_location('integrated_collector',integration_root/'app_gallery/spur-review/collector.py')
integrated_collector=importlib.util.module_from_spec(collector_spec)
collector_spec.loader.exec_module(integrated_collector)
consumer_request={**real_request,'request_id':str(uuid.uuid4())}
consumer_packet=integrated_collector.collect_review('/Users/kevintruong/project/duckdb',consumer_request,{
 'graph':{'status':'ready','artifact':qualified_bytes},
 'history':{'status':'ready','artifact':json.dumps(history_data,sort_keys=True).encode()}})
consumer_table=pa.table({'run_id':[consumer_packet['run_id']],'packet_json':[json.dumps(consumer_packet)]})
with pa.OSFile(str(candidate_root/'review-packet.arrow'),'wb') as target:
 with pa.ipc.new_file(target,consumer_table.schema) as writer: writer.write_table(consumer_table)
(candidate_root/'review-request.json').write_text(json.dumps(consumer_request))
consumer_js = """
import {tableFromIPC} from __ARROW__;
import {acceptReviewRow,selectReview} from __CONSUMER__;
import {renderReviewApp} from __RENDER__;
const root=__ROOT__;
const table=tableFromIPC(await Deno.readFile(root+"/review-packet.arrow"));
if(table.numRows!==1) throw Error("expected exactly one review row");
const expected=JSON.parse(await Deno.readTextFile(root+"/review-request.json"));
let state=acceptReviewRow(table.get(0),expected);
state=selectReview(state,{lens:"architecture",symbol:"HasUniqueGroups"});
const fresh=structuredClone(state.packet);
fresh.request.request_id="replacement-"+expected.request_id;
fresh.run_id="a".repeat(64);
const next=acceptReviewRow({run_id:fresh.run_id,packet_json:JSON.stringify(fresh)},fresh.request,state);
if(JSON.stringify(next.selection)!==JSON.stringify(state.selection)) throw Error("selection lost");
let rejected=false;
try {acceptReviewRow(table.get(0),fresh.request,next)} catch {rejected=true}
if(!rejected) throw Error("stale packet accepted");
await Deno.writeTextFile(root+"/review.html",renderReviewApp(state));
console.log(JSON.stringify({status:"passed",run_id:state.packet.run_id,request_id:expected.request_id,
 tracked_head:state.packet.tracked_files.length,changed_files:state.files.length,
 stale_rejected:rejected,selection_retained:true,production_bridge_qualified:false}));
"""
for token,path in [('__ARROW__',pathlib.Path('/Users/kevintruong/Applications/SpurLab.app/Contents/Resources/deno-host-runtime/arrow.mjs')),('__CONSUMER__',consumer_root/'app_gallery/spur-review/consumer.js'),('__RENDER__',consumer_root/'app_gallery/spur-review/render.js')]:
 consumer_js=consumer_js.replace(token,json.dumps(path.as_uri()))
consumer_js=consumer_js.replace('__ROOT__',json.dumps(str(candidate_root)))
(candidate_root/'consume-real-packet.js').write_text(consumer_js)
consumer_command=['/Users/kevintruong/Applications/SpurLab.app/Contents/MacOS/deno','run','--no-config','--no-lock',
 '--allow-read='+','.join(map(str,[consumer_root,candidate_root,pathlib.Path('/Users/kevintruong/Applications/SpurLab.app')])),
 '--allow-write='+str(candidate_root),str(candidate_root/'consume-real-packet.js')]
consumer_run=subprocess.run(consumer_command,cwd=consumer_root,capture_output=True,text=True)
consumer_receipt={'command':consumer_command,'exit':consumer_run.returncode,'stdout':consumer_run.stdout,'stderr':consumer_run.stderr,
 'ipc_sha256':hashlib.sha256((candidate_root/'review-packet.arrow').read_bytes()).hexdigest(),
 'installed_app_updated':False,'production_bridge_qualified':False}
(candidate_root/'consumer-experiment.json').write_text(json.dumps(consumer_receipt,indent=2))
print(consumer_run.stdout or consumer_run.stderr)
assert consumer_run.returncode==0,consumer_run.stderr


{"status":"passed","run_id":"a494553e5ee4d5614eff09f890d63d0f0a5472d0139291e573066ac58dfe2746","request_id":"55d769fd-149f-4d1a-8c2c-02618eacf0f2","tracked_head":18342,"changed_files":5,"stale_rejected":true,"selection_retained":true,"production_bridge_qualified":false}



In [29]:
# Independent consumer TDD regression verification on final frozen source.
consumer_test_command=['/Users/kevintruong/Applications/SpurLab.app/Contents/MacOS/deno','test','--no-config','--no-lock','app_gallery/spur-review/tests/test_consumer.js']
consumer_tests=subprocess.run(consumer_test_command,cwd=consumer_root,capture_output=True,text=True)
consumer_test_receipt={'command':consumer_test_command,'cwd':str(consumer_root),'exit':consumer_tests.returncode,'stdout':consumer_tests.stdout,'stderr':consumer_tests.stderr,'consumer_sha256':hashlib.sha256((consumer_root/'app_gallery/spur-review/consumer.js').read_bytes()).hexdigest()}
(candidate_root/'consumer-tests-independent.json').write_text(json.dumps(consumer_test_receipt,indent=2))
print(consumer_tests.stdout[-450:])
assert consumer_tests.returncode==0,consumer_tests.stderr
print(json.dumps(json.loads((candidate_root/'browser-receipt.json').read_text()),indent=2))

t hide unavailable or missing providers ... ok (0ms)
changed file inventory survives a curated graph subset ... ok (0ms)
graph display metadata cannot invent a changed file ... ok (0ms)
optional display arrays must be typed before acceptance ... ok (0ms)

ok | 17 passed | 0 failed (24ms)


{
  "status": "passed",
  "checks": [
    "Real packet contains the full tracked inventory and five changed files",
    "Partial semantic graph scope is inspectable",
    "People lens retains the current unknown assignments",
    "Path list renders actual context witnesses",
    "Selected evidence source remains inspectable",
    "Matching refreshed packet retains selection",
    "Superseded packet is rejected without changing displayed run",
    "Selection persists across view reload",
    "Layout fits 390px",
    "Layout fits 768px",
    "Layout fits 1100px",
    "No browser JavaScript exceptions"
  ],
  "errors": [],
  "scope": "Actual notebook-generated HTML in isolated local headless Chrome; nat